In [ ]:
# Cek apakah GPU T4 sudah aktif
import torch
if torch.cuda.is_available():
    print(f"✅ GPU Berhasil Terdeteksi: {torch.cuda.get_device_name(0)}")
else:
    print("❌ GPU Belum aktif! Buka Runtime -> Change runtime type -> Pilih T4 GPU")

# Install semua library yang diperlukan
!pip install -q transformers datasets accelerate scikit-learn openpyxl evaluate

✅ GPU Berhasil Terdeteksi: Tesla T4


In [ ]:
import zipfile
import os
from google.colab import files

# Cek apakah file sudah di-drag & drop langsung atau perlu diupload
if not os.path.exists("berita.zip"):
    print("Silakan pilih dan upload file 'berita.zip':")
    uploaded = files.upload()

# Ekstrak file berita.zip
print("Mengekstrak file berita.zip...")
with zipfile.ZipFile("berita.zip", 'r') as zip_ref:
    zip_ref.extractall(".")

# Verifikasi folder Cleaned
if os.path.exists("Cleaned"):
    print("✅ BERHASIL! Folder 'Cleaned' siap digunakan.")
    print("Isi folder Cleaned:", os.listdir("Cleaned"))
else:
    print("⚠️ Folder Cleaned belum terdeteksi di root, cek letak folder hasil ekstrak.")

Silakan pilih dan upload file 'berita.zip':


Saving berita.zip to berita.zip
Mengekstrak file berita.zip...
✅ BERHASIL! Folder 'Cleaned' siap digunakan.
Isi folder Cleaned: ['dataset_tempo_6k_cleaned.xlsx', 'dataset_cnn_10k_cleaned.xlsx', 'dataset_turnbackhoax_10_cleaned.xlsx', 'dataset_kompas_4k_cleaned.xlsx']


In [ ]:
import pandas as pd
import glob

# 1. BACA SEMUA FILE DARI FOLDER 'Cleaned'
daftar_file = glob.glob("Cleaned/*.xlsx") + glob.glob("Cleaned/*.csv")
dataframes = []

print(f"File yang diproses dari folder Cleaned: {daftar_file}\n")

for f in daftar_file:
    nama_file = f.lower()
    df_temp = pd.read_excel(f) if f.endswith('.xlsx') else pd.read_csv(f)

    # Deteksi otomatis nama kolom teks (FullText, Clean Narasi, dll.)
    kolom_teks = None
    for kandidat in ['fulltext', 'clean narasi', 'narasi', 'cleaned', 'text', 'title', 'berita', 'isi']:
        for col in df_temp.columns:
            if kandidat == str(col).lower().strip():
                kolom_teks = col
                break
        if kolom_teks:
            break

    if not kolom_teks:
        kolom_teks = df_temp.select_dtypes(include='object').columns[0]

    print(f"-> Membaca {f} (Kolom: '{kolom_teks}')")

    # Penentuan Label: Turnbackhoax = 1 (Hoaks), Tempo/CNN/Kompas = 0 (Fakta)
    if 'turnbackhoax' in nama_file:
        df_temp['label'] = 1
    else:
        df_temp['label'] = 0

    df_extracted = pd.DataFrame({
        'text': df_temp[kolom_teks].astype(str),
        'label': df_temp['label'].astype(int)
    })

    dataframes.append(df_extracted)
    print(f"   Jumlah data: {len(df_extracted)} baris | Label: {df_temp['label'].iloc[0]}")

# 2. GABUNGKAN SEMUA SUMBER DARI FOLDER CLEANED
df_gabungan = pd.concat(dataframes, ignore_index=True)

# 3. PEMBERSIHAN DUPLIKAT DAN NILAI KOSONG
df_gabungan.dropna(subset=['text', 'label'], inplace=True)
df_gabungan.drop_duplicates(subset=['text'], inplace=True)
df_gabungan['label'] = df_gabungan['label'].astype(int)

print("\n" + "="*45)
print(f"SUKSES! TOTAL DATA BERSIH: {len(df_gabungan)} baris")
print("Distribusi Label Siap Pakai:")
print(f"  0 (Fakta / Valid Media) : {(df_gabungan['label'] == 0).sum()} baris")
print(f"  1 (Hoaks TurnBackHoax)  : {(df_gabungan['label'] == 1).sum()} baris")
print("="*45)

File yang diproses dari folder Cleaned: ['Cleaned/dataset_tempo_6k_cleaned.xlsx', 'Cleaned/dataset_cnn_10k_cleaned.xlsx', 'Cleaned/dataset_turnbackhoax_10_cleaned.xlsx', 'Cleaned/dataset_kompas_4k_cleaned.xlsx']

-> Membaca Cleaned/dataset_tempo_6k_cleaned.xlsx (Kolom: 'FullText')
   Jumlah data: 6592 baris | Label: 0
-> Membaca Cleaned/dataset_cnn_10k_cleaned.xlsx (Kolom: 'FullText')
   Jumlah data: 9630 baris | Label: 0
-> Membaca Cleaned/dataset_turnbackhoax_10_cleaned.xlsx (Kolom: 'FullText')
   Jumlah data: 10381 baris | Label: 1
-> Membaca Cleaned/dataset_kompas_4k_cleaned.xlsx (Kolom: 'FullText')
   Jumlah data: 4750 baris | Label: 0

SUKSES! TOTAL DATA BERSIH: 31310 baris
Distribusi Label Siap Pakai:
  0 (Fakta / Valid Media) : 20945 baris
  1 (Hoaks TurnBackHoax)  : 10365 baris


In [ ]:
import re

def bersihkan_teks(teks):
    if not isinstance(teks, str):
        return ""
    # Hapus URL web
    teks = re.sub(r'http\S+|www\S+|https\S+', '', teks, flags=re.MULTILINE)
    # Hapus mention @ dan hashtag #
    teks = re.sub(r'(@\w+|#\w+)', '', teks)
    # Hapus tag redaksi berita seperti [CNN], (detikcom), dll.
    teks = re.sub(r'\[.*?\]|\(.*?\)', '', teks)
    # Hapus karakter selain huruf dan angka
    teks = re.sub(r'[^\w\s]', ' ', teks)
    # Hapus spasi berlebih
    teks = re.sub(r'\s+', ' ', teks).strip()
    return teks.lower()

print("Sedang membersihkan teks berita...")
df_gabungan['clean_text'] = df_gabungan['text'].apply(bersihkan_teks)

# Filter berita yang terlalu pendek (di bawah 20 karakter)
df_final = df_gabungan[df_gabungan['clean_text'].str.len() > 20].reset_index(drop=True)
print(f"✅ Selesai! Sebanyak {len(df_final)} baris teks siap diproses ke IndoBERT.")

Sedang membersihkan teks berita...
✅ Selesai! Sebanyak 31303 baris teks siap diproses ke IndoBERT.


In [ ]:
from sklearn.model_selection import train_test_split
from transformers import AutoTokenizer
from datasets import Dataset

# 1. Bagi dataset (80% Train, 10% Validasi, 10% Uji)
train_df, temp_df = train_test_split(df_final, test_size=0.2, random_state=42, stratify=df_final['label'])
val_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=42, stratify=temp_df['label'])

print(f"Data Latih (Train) : {len(train_df)}")
print(f"Data Validasi (Val): {len(val_df)}")
print(f"Data Uji (Test)    : {len(test_df)}")

# 2. Panggil Tokenizer Resmi IndoBERT
MODEL_NAME = "indobenchmark/indobert-base-p1"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def fungsi_tokenisasi(batch):
    # max_length=128 sangat ideal untuk komputasi cepat dan akurat
    return tokenizer(batch['clean_text'], padding="max_length", truncation=True, max_length=128)

print("\nSedang melakukan tokenisasi teks...")
train_dataset = Dataset.from_pandas(train_df).map(fungsi_tokenisasi, batched=True)
val_dataset = Dataset.from_pandas(val_df).map(fungsi_tokenisasi, batched=True)
test_dataset = Dataset.from_pandas(test_df).map(fungsi_tokenisasi, batched=True)
print("✅ Tokenisasi selesai!")

Data Latih (Train) : 25042
Data Validasi (Val): 3130
Data Uji (Test)    : 3131


config.json:   0%|          | 0.00/1.53k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.00 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/229k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]


Sedang melakukan tokenisasi teks...


Map:   0%|          | 0/25042 [00:00<?, ? examples/s]

Map:   0%|          | 0/3130 [00:00<?, ? examples/s]

Map:   0%|          | 0/3131 [00:00<?, ? examples/s]

✅ Tokenisasi selesai!


In [ ]:
import numpy as np
import evaluate
import torch
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer

# 1. Inisialisasi Model IndoBERT untuk 2 Kelas (0: Fakta, 1: Hoaks)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)

# 2. Metrik Pengujian
acc_metric = evaluate.load("accuracy")
f1_metric = evaluate.load("f1")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    acc = acc_metric.compute(predictions=preds, references=labels)['accuracy']
    f1 = f1_metric.compute(predictions=preds, references=labels, average="weighted")['f1']
    return {"accuracy": acc, "f1": f1}

# 3. Konfigurasi Training Optimal untuk Google Colab GPU
training_args = TrainingArguments(
    output_dir="./hasil_training",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    num_train_epochs=3,              # 3 epoch sudah sangat konvergen
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    fp16=True,                       # Mengaktifkan akselerasi GPU Colab
    logging_steps=50,
    save_total_limit=1
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
)

print("🚀 Memulai proses pelatihan model (estimasi 4-7 menit)...")
trainer.train()

[transformers] You passed `num_labels=2` which is incompatible to the `id2label` map of length `5`.


pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  498MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

model.safetensors: reconstructing file:   0%|          |  0.00B /  498MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

... (3 outputs left out)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=2349, training_loss=0.006227304078674237, metrics={'train_runtime': 682.8199, 'train_samples_per_second': 110.023, 'train_steps_per_second': 3.44, 'total_flos': 4941620286243840.0, 'train_loss': 0.006227304078674237, 'epoch': 3.0})

In [ ]:
# 1. Evaluasi pada Test Set (Data murni yang belum pernah dilihat model)
hasil_tes = trainer.evaluate(test_dataset)
print("="*40)
print("HASIL EVALUASI AKHIR (TEST SET):")
print(f"Akurasi : {hasil_tes['eval_accuracy']:.2%}")
print(f"F1-Score: {hasil_tes['eval_f1']:.4f}")
print("="*40)

# 2. Simpan Model Terbaik
model.save_pretrained("./model_hoaks_indobert")
tokenizer.save_pretrained("./model_hoaks_indobert")
print("✅ Model berhasil disimpan ke folder './model_hoaks_indobert'")

# 3. Pengujian Prediksi Nyata (Inference)
from transformers import pipeline

detektor = pipeline("text-classification", model="./model_hoaks_indobert", tokenizer=tokenizer)

# Masukkan contoh kalimat uji coba di bawah ini:
contoh_berita = "Menteri Kesehatan bagikan uang tunai 10 juta bagi pemilik KTP aktif melalui tautan telegram berikut!"
hasil = detektor(bersihkan_teks(contoh_berita))[0]

label_nama = {"LABEL_0": "FAKTA / VALID", "LABEL_1": "HOAKS / MISINFORMASI"}
print("\n--- SIMULASI DETEKSI SISTEM ---")
print(f"Teks Berita : \"{contoh_berita}\"")
print(f"Prediksi    : {label_nama[hasil['label']]}")
print(f"Tingkat Yakin: {hasil['score'] * 100:.2f}%")

Training Loss,Validation Loss,Epoch,Accuracy,F1
0.000041,0.002306,3,0.999681,0.999681


HASIL EVALUASI AKHIR (TEST SET):
Akurasi : 99.97%
F1-Score: 0.9997


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

✅ Model berhasil disimpan ke folder './model_hoaks_indobert'


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]


--- SIMULASI DETEKSI SISTEM ---
Teks Berita : "Menteri Kesehatan bagikan uang tunai 10 juta bagi pemilik KTP aktif melalui tautan telegram berikut!"
Prediksi    : HOAKS / MISINFORMASI
Tingkat Yakin: 100.00%


In [ ]:
import shutil
from google.colab import files

print("Sedang mengompres folder model menjadi file zip...")
shutil.make_archive("model_hoaks_indobert", 'zip', "./model_hoaks_indobert")

print("Mengunduh file ke laptop...")
files.download("model_hoaks_indobert.zip")
print("Unduhan telah dimulai di browser!")

Sedang mengompres folder model menjadi file zip...
Mengunduh file ke laptop...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Unduhan telah dimulai di browser!
